In [ ]:
# T2_V0

import os
import ast

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SEED = 42
DATA = Path('/bohr/train-wfpt/v1/')
WORK = Path('')


In [ ]:
def to_DEVICE(x, device=DEVICE):
    if isinstance(x, dict): return {k:  to_DEVICE(v, device) for k, v in x.items()}
    if isinstance(x, list): return [    to_DEVICE(v, device) for    v in x]
    return x.detach().to(DEVICE)

def train_loop(self, epoch):
    total_pred = []
    total_loss = torch.zeros(2)
    self.model.train()
    pbar = tqdm(self.l_train, desc=f'- Epoch: {epoch:4d}')
    for x, y in pbar:
        x, y = to_DEVICE(x), to_DEVICE(y)
        self.optimizer.zero_grad()
        with torch.amp.autocast(DEVICE):
            pred = self.model(x)
            loss = self.criterion(pred, y)
        self.scaler.scale(loss).backward()
        self.scaler.step (self .optimizer)
        self.scaler.update()
        self.scheduler.step()
        total_pred.append(to_DEVICE(pred, 'cpu'))
        total_loss += torch.tensor([loss.item(), 1.0])
        pbar.set_postfix({
            'Loss': f'{total_loss[0] / total_loss[1]:.6f}',
        })
    loss = total_loss[0] / total_loss[1]
    return total_pred, loss

@torch.no_grad()
def valid_loop(self, epoch):
    total_pred = []
    total_loss = torch.zeros(2)
    total_rate = torch.zeros(2)
    self.model.eval()
    pbar = tqdm(self.l_valid, desc=f'  Epoch: {epoch:4d}')
    for x, y in pbar:
        x, y = to_DEVICE(x), to_DEVICE(y)
        with torch.amp.autocast(DEVICE):
            pred = self.model(x)
            loss = self.criterion(pred, y)
        total_pred.append(to_DEVICE(pred, 'cpu'))
        total_loss += torch.tensor([loss.item(), 1.0])
        total_rate += self.metrics( pred, y).cpu()
        pbar.set_postfix({
            'Loss': f'{total_loss[0] / total_loss[1]:.6f}',
            'Rate': f'{total_rate[0] / total_rate[1]:.6f}',
        })
    loss = total_loss[0] / total_loss[1]
    rate = total_rate[0] / total_rate[1]
    return total_pred, loss, rate

@torch.no_grad()
def infer_loop(self, l_infer):
    total_pred = []
    self.model.eval()
    for x, y in tqdm(l_infer):
        x, y = to_DEVICE(x), y
        with  torch.amp.autocast(DEVICE):
            total_pred.append(to_DEVICE(self.model(x), 'cpu'))
    return  total_pred


In [ ]:
def trainRunner(self):
    final_rate =-1e18
    final_loss = 1e18
    for i in range(1, self.config.epoch + 1):
        train_pred, train_loss              = train_loop(self, i)
        valid_pred, valid_loss, valid_rate   = valid_loop(self, i)
        if (final_rate,-final_loss) < (valid_rate,-valid_loss):
            final_rate, final_loss  =  valid_rate, valid_loss
            torch.save(self.model.state_dict(), WORK / f'{self.config.stage}.pth')
            torch.save(train_pred, WORK / f'{self.config.stage}_train.pt')
            torch.save(valid_pred, WORK / f'{self.config.stage}_valid.pt')
    return final_rate, final_loss

def inferRunner(self):
    param = torch.load(WORK / f'{self.config.stage}.pth', map_location=DEVICE)
    self.model.load_state_dict(param)
    torch.save(infer_loop(self, self.l_testA), WORK / f'{self.config.stage}_testA.pt')
    torch.save(infer_loop(self, self.l_testB), WORK / f'{self.config.stage}_testB.pt')


In [ ]:
def parse_vector(value):
    try:
        # 检查输入是否为 None 或空字符串
        if value is None or str(value).strip() == "":
            return np.zeros(6, dtype=np.float32)
        
        # 使用 literal_eval 解析输入
        vector = ast.literal_eval(str(value).strip())
        
        # 检查解析结果是否为有效的列表或元组
        if not isinstance(vector, (list, tuple)) or not vector:
            raise ValueError(f"Invalid vector value: {value}")
        
        # 转换为 NumPy 数组并返回
        return np.asarray([float(item) for item in vector], dtype=np.float32)
    
    except (ValueError, SyntaxError) as e:
        print(f"Error parsing vector: {e}")
        return np.zeros(6, dtype=np.float32)  # 返回默认值
    
def vector_to_string(vector):
    # 检查输入是否为 6 维 NumPy 数组
    if not isinstance(vector, np.ndarray) or vector.shape != (6,):
        raise ValueError("Input must be a 6-dimensional NumPy array.")
    
    # 使用 repr() 将数组转换为字符串
    return repr(vector.tolist())

def process_dataframe(df):
    result = {}
    grouped = df.groupby('task_index')
    for task_index, group in grouped:
        task_list = []
        for _, row in group.iterrows():
            task_entry = {
                'index': row['index'],
                'frame_index': row['frame_index'],
                'timestamp': row['timestamp'],
                'simulation_positions': parse_vector(row['simulation_positions']),
                'observation_state': parse_vector(row['observation_state']) if 'observation_state' in row else np.zeros(6, dtype=np.float32)
            }
            task_list.append(task_entry)
        result[task_index] = task_list
    return result


In [ ]:
class Nueryim_TS(Dataset):
    def __init__(self, task_data):
        self.data = task_data
    def __len__(self):
        return  len(self. data)  # 返回完整的长度
    def __getitem__(self, indx):
        x = torch.zeros(18, dtype=torch.float32)
        if indx >= 0: x[12:18] = torch.tensor(self.data[indx - 0]['simulation_positions'], dtype=torch.float32)
        if indx >= 1: x[ 6:12] = torch.tensor(self.data[indx - 1]['simulation_positions'], dtype=torch.float32)
        if indx >= 2: x[ 0: 6] = torch.tensor(self.data[indx - 2]['simulation_positions'], dtype=torch.float32)
        y = torch.tensor(self.data[indx]['observation_state'], dtype=torch.float32)
        return x, y


In [ ]:
class Nueryim_NN(nn.Module):
    def __init__(self, I=18, H=256, O=6, P=0.2):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(I, H), nn.LeakyReLU(inplace=True), nn.Dropout(P),
            nn.Linear(H, H), nn.LeakyReLU(inplace=True), nn.Dropout(P),
            nn.Linear(H, H), nn.LeakyReLU(inplace=True), nn.Dropout(P),
            nn.Linear(H, O)
        )
    def forward(self, x):
        x = self.head(x)
        return x


In [ ]:
class Nueryim_LS(nn.Module):
    def __init__(self):
        super().__init__()
        self.func = nn.L1Loss()
    def forward(self, pred, gold):
        return  self. func (pred, gold)

class Nueryim_CS(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, pred, gold):
        return torch.tensor([0.0, 1.0])


In [ ]:
class Nueryim_TT:
    class config:
        stage = 'T2_V0'
        epoch = 32
        batch = 256
        lrate = 1e-3
        lrmax = 5e-3
        decay = 1e-4
    d_total = process_dataframe(pd.read_csv(DATA / 'train.csv'))
    s_total = []
    for k, v in d_total.items():
        s_total.append(Nueryim_TS(v))
    split = int(len(s_total) * 0.8)
    s_train = ConcatDataset(s_total[:split ])
    s_valid = ConcatDataset(s_total[ split:])
    l_train = DataLoader(s_train, config.batch, True , num_workers=4, prefetch_factor=2)
    l_valid = DataLoader(s_valid, config.batch, False, num_workers=4, prefetch_factor=2)
    scaler = torch.amp.GradScaler(DEVICE)
    model       = Nueryim_NN().to(DEVICE)
    metrics     = Nueryim_CS().to(DEVICE)
    criterion   = Nueryim_LS().to(DEVICE)
    optimizer   = torch.optim.AdamW(
        model.parameters(),
        lr          =config.lrate,
        weight_decay=config.decay,
    )
    scheduler   = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr      =config.lrmax,
        total_steps =config.epoch * len(l_train),
    )

trainRunner(Nueryim_TT)

In [ ]:
HIDE = Path(os.environ.get('DATA_PATH'))

class Nueryim_II(Nueryim_TT):
    class config(Nueryim_TT.config):
        batch = 256
    d_testA = process_dataframe(pd.read_csv(HIDE / 'val.csv'))
    d_testB = process_dataframe(pd.read_csv(HIDE / 'test.csv'))
    s_testA = ConcatDataset([Nueryim_TS(v) for k, v in d_testA.items()])
    s_testB = ConcatDataset([Nueryim_TS(v) for k, v in d_testB.items()])
    l_testA = DataLoader(s_testA, config.batch, False, num_workers=4, prefetch_factor=2)
    l_testB = DataLoader(s_testB, config.batch, False, num_workers=4, prefetch_factor=2)

inferRunner(Nueryim_II)

In [ ]:


def calc_back(data, pred):
    pred = torch.cat(pred, dim=0)
    mask = [False] * len(data)
    for indx in range(len(data)):
        if data.at[indx, 'observation_state'] is None or pd.isnull(data.at[indx, 'observation_state']):
            data.at[indx, 'observation_state'] = vector_to_string(pred[indx].float().numpy())
            mask[indx] = True  # 标记该行需要保留
    return data[mask].drop(['simulation_positions'], axis=1)


In [ ]:
calc_back(pd.read_csv(HIDE / 'val.csv' ), torch.load(WORK / f'{Nueryim_II.config.stage}_testA.pt')).to_csv('submission_val.csv' , index=False)
calc_back(pd.read_csv(HIDE / 'test.csv'), torch.load(WORK / f'{Nueryim_II.config.stage}_testB.pt')).to_csv('submission_test.csv', index=False)


In [ ]:
import zipfile

zip_path = "submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("submission_val.csv")
    zf.write("submission_test.csv")